# Notebook 01 — Environment Setup

**Duration:** ~20 minutes

Welcome. In this notebook you will set up the tools we use throughout the workshop.

By the end you will have:
- Python running in VS Code on your own laptop
- A working connection to the university machine that runs our large language model (LLM) — an AI system that can read and generate text
- Enough confidence to keep going

You installed the tools before the workshop, using the setup guide. Everything runs here in **VS Code** — a free program for writing and running code. Your Python code runs on your own laptop.

## What is a cell?

This notebook is made of **cells**. Each cell is a block that contains either text (like this one) or code (like the one below).

To run a code cell:
1. Click on it so it is highlighted
2. Press **Shift + Enter** on your keyboard (or click the ▶ play button on the left)

The first time you run a cell, VS Code may ask you to **select a kernel**. A kernel is the program that runs your Python code. Choose the one called **.venv**. You only need to do this once per notebook.

The result appears directly below the cell. If something goes wrong, you will see an error message — that is normal. Read the last line of the error. It usually tells you what happened.

Let's try it.

## Exercise 1 — Python as a calculator

Python can do arithmetic. Run the cell below to see it work.

In [1]:
2 + 3

5

You should see the number `5` below the cell.

Now try some of these. Change the numbers, or write your own. Run the cell each time to see the result.

In [ ]:
# Try any of these — change the numbers, or write your own.
# Delete the # at the start of a line to "uncomment" it and make it run.

# 10 * 5
# 100 / 3
# 2 ** 10        # ** means "to the power of"
# (4 + 6) * 2

Lines that start with `#` are **comments**. They are notes for us to understand what the code is doing, Python ignores them.

## Exercise 2 — Variables and print()

A **variable** stores a value so you can use it later. Think of it as a labelled box: you put something in, and refer to it by name.

In [2]:
name = "Toby"

In [3]:
print(name)

Toby


In [4]:
# The `print()` function displays a value on screen. Run this cell to see both in action.
print("Hello, my name is", name)
print("Welcome to the workshop.")

Hello, my name is Toby
Welcome to the workshop.


Your turn. Change `"____"` to your own name and run the cell.

In [5]:
name = "____"    # <-- replace ____ with your name (keep the quotes)
print("Hello my name is", name)

# You can store numbers too:
sections = 200
print("The Privacy Act 2020 has roughly", sections, "sections.")
print("Reading them all would take a while. That is why we have an LLM.")

Hello my name is ____
The Privacy Act 2020 has roughly 200 sections.
Reading them all would take a while. That is why we have an LLM.


Notice: text goes in quotes (`"like this"`), numbers do not. If you see an error about quotes, check that both the opening and closing quotes are there.

## Setting up our tools

The large language model (LLM) we use today runs on a **Dell Pro Max with GB10**. This is a compact desktop computer built for running AI models. We have several of these machines, and each one runs its own copy of an LLM.

To connect to one of these machines, you need two things:

- A **base URL** — the web address of the machine you are connecting to.
- An **API key** — a password that identifies you to that machine.

Each machine has its own base URL and API key. The pair you use decides which machine answers your questions. You will get yours in the next step.

<details>
<summary><strong>What is vLLM & an API?</strong></summary>

**vLLM** is free, open-source software that runs LLMs. It loads the model onto the machine and answers questions sent to it. It can handle requests from many people at the same time. The models themselves are built by other organisations. vLLM is the software that runs them on our hardware.

An **API** (Application Programming Interface) is a way for one program to talk to another over the internet. Instead of opening a chatbot in your browser, our Python code sends a question directly to the model and receives a structured answer back.

To send our questions, we use a Python library called `openai`. Despite the name, nothing is sent to the company OpenAI. The library follows a standard format for talking to LLMs, and vLLM understands that format. That means we only need to tell the library our base URL and API key. This also means that once we have written some code using the `openai` library, it will work for a number of different AI providers. For example, we can use the same code, with minor tweaks, to connect to OpenAI, Groq, Mistral AI, Google AI platform, 
Anthropic, Ollama, and more! The only thing we would need to change is the **base URL** and **API key** and instead of firing our requests off to the university infrastructure, we would be firing them off to the relevant organisation.

Here is how the pieces fit together:

```
┌──────────────────┐                  ┌──────────────────┐
│  Your laptop     │                  │  Dell Pro Max    │
│                  │  API call over   │  (runs vLLM)     │
│  VS Code runs    │ ──────────────── │                  │
│  your Python     │  university VPN  │  The LLM         │
└──────────────────┘                  └──────────────────┘
         │                                     │
    You write code                      Model generates
    and press Run                       a response
         │                                     │
         └──────────── answer arrives ─────────┘
```

Your code runs on your own laptop. The LLM lives on a Dell Pro Max inside the university network. To reach it, your laptop must be connected to the university **VPN** (virtual private network) — a secure connection into the university network from outside it.

Your code sends a question to the Dell Pro Max, using the base URL and API key, and the answer comes back. Your questions and data stay between your laptop and university hardware. Nothing is sent to an outside AI company.

</details>


## Get your base URL and API key

On the day, your instructor will give you a **base URL** and an **API key** for one of the Dell Pro Max machines.

**Important:** Your API key is a password. Do not share it outside the workshop or post it online.

## Store them in a .env file

Instead of pasting your key into every notebook, we store it once in a small text file called `.env` (say "dot env"). Every notebook in this workshop reads it automatically. It also keeps your key out of the notebooks, so you can share a notebook without sharing your key.

1. In VS Code, open the **Explorer** — the list of files on the left. (If you cannot see it, click the top icon in the far-left bar.)
2. Find the file called `.env.example`. It is in the main workshop folder, not inside the `notebooks` folder.
3. Right-click it, choose **Rename**, and change the name to exactly `.env`
4. Click `.env` to open it. Paste your base URL and API key after the `=` signs, like this:

```
LLM_BASE_URL=paste_your_base_url_here
LLM_API_KEY=paste_your_api_key_here
```

5. Save the file: **Ctrl + S** on Windows, **Cmd + S** on Mac.

Do not add quote marks or spaces around the values.

Finally, check that you are connected to the university VPN. Then run the setup cell below.

In [6]:
# ============================================================
# SETUP CELL — Run this once at the start of every notebook
# ============================================================

import os, json, base64, requests, io
from dotenv import load_dotenv
from openai import OpenAI
from lxml import etree
from PIL import Image
from IPython.display import Image as IPImage, display

# Load your base URL and API key from the .env file (set up above)
load_dotenv(override=True)
if not os.getenv("LLM_BASE_URL") or not os.getenv("LLM_API_KEY"):
    raise RuntimeError("Could not find LLM_BASE_URL or LLM_API_KEY. Check your .env file (see notebook 01).")

client = OpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])

# Each Dell Pro Max machine serves one model. This is its name.
# It reads both text and images.
MODEL = "nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-NVFP4"

# Turn off the model's "thinking" step. Answers come back several times faster,
# which matters when many people share one machine.
NO_THINKING = {"chat_template_kwargs": {"enable_thinking": False}}

print(f"Using model {MODEL}.")
print("Setup complete.")

Using model nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-NVFP4.
Setup complete.


You should see `Using model` followed by the name of the model, then `Setup complete.`

<details>
<summary><strong>Got an error? Click here for troubleshooting.</strong></summary>

If you see `Could not find LLM_BASE_URL or LLM_API_KEY`, check:
- Is the file named exactly `.env`? Not `.env.example`, `env`, or `.env.txt`.
- Is it in the main workshop folder, not inside the `notebooks` folder?
- Did you save the file after pasting your values?

If you see `ModuleNotFoundError`, VS Code is using the wrong kernel. Click the kernel name in the top-right corner of the notebook and choose **.venv**.

After fixing anything, run the setup cell again.

</details>

## Exercise 3 — Check the machine is ready

The setup cell only read your `.env` file. It did not contact the Dell Pro Max. Let's do that now.

The cell below sends a small "are you there?" request to the machine. It uses your API key, and asks the machine which model it has loaded. No question is sent to the LLM yet — that is the first thing you will do in notebook 02.


In [ ]:
from openai import AuthenticationError, APIConnectionError

# Ask the machine which models it has loaded. This checks three things at once:
# the base URL is reachable, the API key is accepted, and our model is running.
try:
    loaded_models = [m.id for m in client.models.list().data]
except AuthenticationError:
    raise RuntimeError("The machine rejected your API key. Check LLM_API_KEY in your .env file.") from None
except APIConnectionError:
    raise RuntimeError(
        "Could not reach the machine. Check that you are on the university VPN "
        "and that LLM_BASE_URL in your .env file is correct."
    ) from None

if MODEL in loaded_models:
    print("The machine is ready.")
    print(f"It has {MODEL} loaded and waiting for you.")
else:
    print("The machine answered, but it does not have the model we expected.")
    print("It reports:", loaded_models)
    print("Tell your instructor.")


You should see `The machine is ready.` followed by the model name.

<details>
<summary><strong>Got an error? Click here for troubleshooting.</strong></summary>

If you see `Could not reach the machine`, check:
- Are you connected to the university VPN?
- Did you copy the full base URL, with no extra spaces?

If you see `The machine rejected your API key`, check:
- Did you copy the full API key, with no extra spaces?
- Did you save the `.env` file after pasting?

After fixing anything, run the setup cell again, then this cell.

</details>


## What you accomplished

In this notebook you:

- Ran Python code on your own laptop using VS Code
- Learned what cells, variables, and `print()` do
- Stored your base URL and API key in a `.env` file
- Confirmed that the Dell Pro Max accepts your key and has the LLM loaded

This is the ingestion infrastructure — Step 1 of the research workflow we introduced earlier. Everything else builds on what you just set up.

**Next up:** In notebook 02, you send your first message to the model. Then we look more closely at how it behaves — why it sometimes gives different answers to the same question, and where its knowledge breaks down.